### Simulate Streaming from Folder (Delta as Source)

In [0]:
stream_df = spark.readStream \
    .format("delta") \
    .load("/Volumes/workspace/ecommerce/delta/bronze/events")

### Write Streaming Output to Delta (Silver)

In [0]:
from pyspark.sql import functions as F

clean_stream = stream_df.filter(F.col('price') > 0) \
          .dropDuplicates(['user_session', 'event_time']) \
          .filter(F.col('category_code').isNotNull()) \
          .filter(F.col("event_type") == "purchase") \
          .withColumn(
                    "product_category",
                    F.element_at(
                        F.split(F.col("category_code"), "\\."),
                        -1
                    )
                )   

In [0]:
query = clean_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/ecommerce/checkpoints/purchase_stream_v2") \
    .trigger(availableNow=True) \
    .start("/Volumes/workspace/ecommerce/delta/silver/purchase_events_v2")

query.awaitTermination()

### Query Streaming Results

In [0]:
silver_df = spark.read.format("delta") \
    .load("/Volumes/workspace/ecommerce/delta/silver/purchase_events_v2")

silver_df.display()